In [1]:
#!pip install biopython>=1.83 pandas>=2.0.0 matplotlib>=3.7.0 seaborn>=0.12.0 pyarrow>=15.0.0 openpyxl>=3.1.0

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [3]:
from Bio import Entrez
from datetime import datetime, timedelta
import pandas as pd
import re
import json
from collections import Counter
import time
from urllib.error import URLError, HTTPError

RETRIES = 3
BACKOFF_FACTOR = 5
RATE_LIMIT_DELAY = 0.34  # ~3 requests/sec per NCBI policy  # NEW

# ---------------------------------------------------------------------
# 1. Robust PubMed query builder remains the same
# ---------------------------------------------------------------------
def prepare_antibiotic_pubmed_query(name):
    import re
    parts = re.split(r'[-/]', name)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) > 1:
        and_part = " AND ".join(f'"{p}"[Abstract]' for p in parts)
        query = f'("{name}"[Abstract] OR ({and_part}))'
    else:
        query = f'"{name}"[Abstract]'
    return query


# ---------------------------------------------------------------------
# 2. Patched: search_pmids with explicit error handling, throttling, WebEnv checks
# ---------------------------------------------------------------------
def search_pmids(antibiotic, pathogen, bacteria_synonym_df, extra_clauses,
                 min_date, max_date, email):
    """
    Searches PubMed for PMIDs with retries and exponential backoff.
    Handles HTTP/URL errors, respects NCBI rate limits, and validates WebEnv.
    """
    Entrez.email = email
    query = build_antibiotic_pathogen_query(
        antibiotic, pathogen, bacteria_synonym_df,
        extra_clauses=extra_clauses,
        min_date=min_date, max_date=max_date
    )

    for i in range(RETRIES):
        try:
            time.sleep(RATE_LIMIT_DELAY)  # NEW – throttle requests
            handle = Entrez.esearch(db="pubmed", term=query, usehistory="y", retmax=0)
            try:
                record = Entrez.read(handle)
            except Exception as e:
                print(f"Failed to parse Entrez response: {e}")  # NEW
                record = {}
            handle.close()

            total_count = int(record.get("Count", 0))
            webenv = record.get("WebEnv")
            query_key = record.get("QueryKey")

            if not webenv or not query_key:
                print("Warning: missing WebEnv or QueryKey — skipping retrieval.")  # NEW
                return query, 0, None, None

            return query, total_count, webenv, query_key

        except (HTTPError, URLError) as e:
            delay = BACKOFF_FACTOR * (2 ** i)
            print(f"Network/HTTP error during Entrez search: {e}. Retrying in {delay} seconds...")  # NEW
            time.sleep(delay)
        except Exception as e:
            print(f"Unexpected error during Entrez search: {e}")
            break

    print(f"Failed to connect to Entrez after {RETRIES} retries.")
    return query, 0, None, None


# ---------------------------------------------------------------------
# 3. Patched: retrieve_pmids with rate limiting and parse guards
# ---------------------------------------------------------------------
def retrieve_pmids(query, webenv, query_key, count, batch_size=5000):
    pmids = []
    if not query or not webenv or not query_key or count == 0:
        return pmids

    for start in range(0, count, batch_size):
        for attempt in range(RETRIES):
            try:
                time.sleep(RATE_LIMIT_DELAY)  # NEW – respect rate limits
                handle = Entrez.esearch(
                    db="pubmed",
                    term=query,
                    usehistory="y",
                    webenv=webenv,
                    query_key=query_key,
                    retstart=start,
                    retmax=batch_size
                )
                try:
                    record = Entrez.read(handle)
                except Exception as e:
                    print(f"Failed to parse Entrez record: {e}")  # NEW
                    record = {}
                handle.close()

                pmids.extend(record.get("IdList", []))
                break  # success, move to next batch
            except (HTTPError, URLError) as e:
                delay = BACKOFF_FACTOR * (2 ** attempt)
                print(f"Network error fetching PMIDs: {e}. Retrying in {delay}s...")  # NEW
                time.sleep(delay)
            except Exception as e:
                print(f"Unexpected error retrieving PMIDs: {e}")
                break

    return pmids


# ---------------------------------------------------------------------
# 4. Patched: fetch_pub_metadata with throttling, error guards, and fallback parsing
# ---------------------------------------------------------------------
def fetch_pub_metadata(pmids, email="your_email@example.com", batch_size=200):
    Entrez.email = email
    date_map = {}
    venue_map = {}

    for start in range(0, len(pmids), batch_size):
        batch = pmids[start:start + batch_size]
        for attempt in range(RETRIES):
            try:
                time.sleep(RATE_LIMIT_DELAY)  # NEW
                handle = Entrez.esummary(db="pubmed", id=",".join(batch))
                try:
                    records = Entrez.read(handle)
                except Exception as e:
                    print(f"Failed to parse Entrez summary: {e}")
                    records = {}
                handle.close()

                if isinstance(records, dict):
                    summaries = records.get("DocumentSummarySet", {}).get("DocumentSummary", [])
                else:
                    summaries = records

                for summary in summaries:
                    pmid_value = summary.get("Id")
                    if pmid_value is None:
                        continue
                    pmid = str(pmid_value)
                    pubdate = summary.get("PubDate") or summary.get("EPubDate") or ""
                    epub = summary.get("EPubDate")
                    venue = summary.get("FullJournalName") or summary.get("Source") or "Unknown"
                    parsed_date = parse_pubdate(pubdate)
                    if not parsed_date and epub:
                        parsed_date = parse_pubdate(epub)
                    if parsed_date:
                        date_map[pmid] = parsed_date
                    venue_map[pmid] = venue
                break  # success
            except (HTTPError, URLError) as e:
                delay = BACKOFF_FACTOR * (2 ** attempt)
                print(f"HTTP/Network error in fetch_pub_metadata: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            except Exception as e:
                print(f"Unexpected error in fetch_pub_metadata: {e}")
                break

    return date_map, venue_map


# ---------------------------------------------------------------------
# 5. Patched: merge_with_existing with safe write retries
# ---------------------------------------------------------------------
def merge_with_existing(new_df, parquet_path="pmids_results.parquet"):
    """
    Merge new results into existing parquet (if present) safely.
    Retries parquet write on transient I/O errors.
    """
    if os.path.exists(parquet_path):
        old_df = pd.read_parquet(parquet_path)
        for df in (old_df, new_df):
            if "venue_histogram" not in df.columns:
                df["venue_histogram"] = json.dumps({})
        merged_df = pd.concat([old_df, new_df], ignore_index=True)
        merged_df = merged_df.drop_duplicates(
            subset=["pathogen", "antibiotic", "start_date", "end_date"],
            keep="last"
        )
    else:
        if "venue_histogram" not in new_df.columns:
            new_df["venue_histogram"] = json.dumps({})
        merged_df = new_df

    # Retry parquet write 3 times on OSError
    for attempt in range(3):  # NEW
        try:
            merged_df.to_parquet(parquet_path, index=False)
            break
        except OSError as e:
            delay = 2 ** attempt
            print(f"File write failed ({e}), retrying in {delay}s...")
            time.sleep(delay)
    return merged_df

from urllib.error import URLError, HTTPError
import time
import json
from collections import Counter
from datetime import datetime, timedelta
import pandas as pd
import os

RETRIES = 3
BACKOFF_FACTOR = 5
RATE_LIMIT_DELAY = 0.34  # ~3 requests/sec per NCBI policy  # NEW

# ---------------------------------------------------------------------
# 1. Robust PubMed query builder remains the same
# ---------------------------------------------------------------------
def prepare_antibiotic_pubmed_query(name):
    import re
    parts = re.split(r'[-/]', name)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) > 1:
        and_part = " AND ".join(f'"{p}"[Abstract]' for p in parts)
        query = f'("{name}"[Abstract] OR ({and_part}))'
    else:
        query = f'"{name}"[Abstract]'
    return query


# ---------------------------------------------------------------------
# 2. Patched: search_pmids with explicit error handling, throttling, WebEnv checks
# ---------------------------------------------------------------------
def search_pmids(antibiotic, pathogen, bacteria_synonym_df, extra_clauses,
                 min_date, max_date, email):
    """
    Searches PubMed for PMIDs with retries and exponential backoff.
    Handles HTTP/URL errors, respects NCBI rate limits, and validates WebEnv.
    """
    Entrez.email = email
    query = build_antibiotic_pathogen_query(
        antibiotic, pathogen, bacteria_synonym_df,
        extra_clauses=extra_clauses,
        min_date=min_date, max_date=max_date
    )

    for i in range(RETRIES):
        try:
            time.sleep(RATE_LIMIT_DELAY)  # NEW – throttle requests
            handle = Entrez.esearch(db="pubmed", term=query, usehistory="y", retmax=0)
            try:
                record = Entrez.read(handle)
            except Exception as e:
                print(f"Failed to parse Entrez response: {e}")  # NEW
                record = {}
            handle.close()

            total_count = int(record.get("Count", 0))
            webenv = record.get("WebEnv")
            query_key = record.get("QueryKey")

            if not webenv or not query_key:
                print("Warning: missing WebEnv or QueryKey — skipping retrieval.")  # NEW
                return query, 0, None, None

            return query, total_count, webenv, query_key

        except (HTTPError, URLError) as e:
            delay = BACKOFF_FACTOR * (2 ** i)
            print(f"Network/HTTP error during Entrez search: {e}. Retrying in {delay} seconds...")  # NEW
            time.sleep(delay)
        except Exception as e:
            print(f"Unexpected error during Entrez search: {e}")
            break

    print(f"Failed to connect to Entrez after {RETRIES} retries.")
    return query, 0, None, None


# ---------------------------------------------------------------------
# 3. Patched: retrieve_pmids with rate limiting and parse guards
# ---------------------------------------------------------------------
def retrieve_pmids(query, webenv, query_key, count, batch_size=5000):
    pmids = []
    if not query or not webenv or not query_key or count == 0:
        return pmids

    for start in range(0, count, batch_size):
        for attempt in range(RETRIES):
            try:
                time.sleep(RATE_LIMIT_DELAY)  # NEW – respect rate limits
                handle = Entrez.esearch(
                    db="pubmed",
                    term=query,
                    usehistory="y",
                    webenv=webenv,
                    query_key=query_key,
                    retstart=start,
                    retmax=batch_size
                )
                try:
                    record = Entrez.read(handle)
                except Exception as e:
                    print(f"Failed to parse Entrez record: {e}")  # NEW
                    record = {}
                handle.close()

                pmids.extend(record.get("IdList", []))
                break  # success, move to next batch
            except (HTTPError, URLError) as e:
                delay = BACKOFF_FACTOR * (2 ** attempt)
                print(f"Network error fetching PMIDs: {e}. Retrying in {delay}s...")  # NEW
                time.sleep(delay)
            except Exception as e:
                print(f"Unexpected error retrieving PMIDs: {e}")
                break

    return pmids


# ---------------------------------------------------------------------
# 4. Patched: fetch_pub_metadata with throttling, error guards, and fallback parsing
# ---------------------------------------------------------------------
def fetch_pub_metadata(pmids, email="your_email@example.com", batch_size=200):
    Entrez.email = email
    date_map = {}
    venue_map = {}

    for start in range(0, len(pmids), batch_size):
        batch = pmids[start:start + batch_size]
        for attempt in range(RETRIES):
            try:
                time.sleep(RATE_LIMIT_DELAY)  # NEW
                handle = Entrez.esummary(db="pubmed", id=",".join(batch))
                try:
                    records = Entrez.read(handle)
                except Exception as e:
                    print(f"Failed to parse Entrez summary: {e}")
                    records = {}
                handle.close()

                if isinstance(records, dict):
                    summaries = records.get("DocumentSummarySet", {}).get("DocumentSummary", [])
                else:
                    summaries = records

                for summary in summaries:
                    pmid_value = summary.get("Id")
                    if pmid_value is None:
                        continue
                    pmid = str(pmid_value)
                    pubdate = summary.get("PubDate") or summary.get("EPubDate") or ""
                    epub = summary.get("EPubDate")
                    venue = summary.get("FullJournalName") or summary.get("Source") or "Unknown"
                    parsed_date = parse_pubdate(pubdate)
                    if not parsed_date and epub:
                        parsed_date = parse_pubdate(epub)
                    if parsed_date:
                        date_map[pmid] = parsed_date
                    venue_map[pmid] = venue
                break  # success
            except (HTTPError, URLError) as e:
                delay = BACKOFF_FACTOR * (2 ** attempt)
                print(f"HTTP/Network error in fetch_pub_metadata: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            except Exception as e:
                print(f"Unexpected error in fetch_pub_metadata: {e}")
                break

    return date_map, venue_map


# ---------------------------------------------------------------------
# 5. Patched: merge_with_existing with safe write retries
# ---------------------------------------------------------------------
def merge_with_existing(new_df, parquet_path="pmids_results.parquet"):
    """
    Merge new results into existing parquet (if present) safely.
    Retries parquet write on transient I/O errors.
    """
    if os.path.exists(parquet_path):
        old_df = pd.read_parquet(parquet_path)
        for df in (old_df, new_df):
            if "venue_histogram" not in df.columns:
                df["venue_histogram"] = json.dumps({})
        merged_df = pd.concat([old_df, new_df], ignore_index=True)
        merged_df = merged_df.drop_duplicates(
            subset=["pathogen", "antibiotic", "start_date", "end_date"],
            keep="last"
        )
    else:
        if "venue_histogram" not in new_df.columns:
            new_df["venue_histogram"] = json.dumps({})
        merged_df = new_df

    # Retry parquet write 3 times on OSError
    for attempt in range(3):  # NEW
        try:
            merged_df.to_parquet(parquet_path, index=False)
            break
        except OSError as e:
            delay = 2 ** attempt
            print(f"File write failed ({e}), retrying in {delay}s...")
            time.sleep(delay)
    return merged_df

def expand_synonyms(pathogen, bacteria_synonym_df):
    """
    Return a *list* of synonyms for `pathogen` (deduped, stripped).
    """
    mask = bacteria_synonym_df['bacteria name'].str.lower() == pathogen.lower()
    # Get values, drop NaN, coerce to str, strip, dedupe to list
    vals = (
        bacteria_synonym_df.loc[mask, 'bacteria synonym']
        .dropna()
        .astype(str)
        .str.strip()
    )
    synonyms = sorted({v for v in vals if v})
    return list(synonyms)


MONTH_LOOKUP = {
    'jan': 1,
    'feb': 2,
    'mar': 3,
    'apr': 4,
    'may': 5,
    'jun': 6,
    'jul': 7,
    'aug': 8,
    'sep': 9,
    'oct': 10,
    'nov': 11,
    'dec': 12,
}

def parse_pubdate(pubdate):
    if not pubdate:
        return None
    text = str(pubdate).strip()
    if not text:
        return None
    match = re.search(r'(\d{4})', text)
    if not match:
        return None
    year = int(match.group(1))
    lowered = text.lower()
    month = 1
    for key, value in MONTH_LOOKUP.items():
        if key in lowered:
            month = value
            break
    try:
        return datetime(year, month, 1)
    except ValueError:
        return datetime(year, 1, 1)

def build_antibiotic_pathogen_query(antibiotic, pathogen, bacteria_synonym_df,
                                    extra_clauses=None, min_date=None, max_date=None):
    # --- Expand pathogen synonyms (as list) ---
    pathogen_variants = expand_synonyms(pathogen, bacteria_synonym_df)

    # Ensure we have something; also include the canonical pathogen once
    if not pathogen_variants:
        pathogen_variants = [pathogen]
    else:
        # Make sure the canonical name is included too
        if pathogen not in pathogen_variants:
            pathogen_variants = [pathogen] + pathogen_variants

    # Build OR clause for all pathogen variants
    pathogen_clause = '(' + ' OR '.join(
        [f'"{p}"[Abstract]' for p in pathogen_variants]
    ) + ')'

    # Antibiotic clause (single or could be extended similarly)
    antibiotic_clause = prepare_antibiotic_pubmed_query(antibiotic)

    term_parts = [f'({antibiotic_clause} AND {pathogen_clause})']

    # Add optional extras
    if extra_clauses:
        term_parts.extend(extra_clauses)

    # Add date range if given
    if min_date and max_date:
        term_parts.append(f'("{min_date}"[Date - Publication] : "{max_date}"[Date - Publication])')

    # Final compound query
    return ' AND '.join(term_parts)




def fetch_pmids_by_date_and_keywords(min_date, max_date, antibiotic, pathogen, bacteria_synonym_df, email="your_email@example.com", batch_size=5000, extra_clauses=None):
    """
    Retrieve PMIDs from PubMed within a date range where all keywords appear in the abstract.
    Recursively splits the date range if a query exceeds PubMed's 9,999 record limit.
    """
    query, count, webenv, query_key = search_pmids(
        antibiotic, 
        pathogen, 
        bacteria_synonym_df,
        extra_clauses=extra_clauses,
        min_date=min_date,
        max_date=max_date,
        email=email
    )
    if count == 0:
        return []
    if count > 9999:
        start_dt = datetime.strptime(min_date, "%Y/%m/%d")
        end_dt = datetime.strptime(max_date, "%Y/%m/%d")
        mid_year = (start_dt.year + end_dt.year) // 2
        mid_dt = datetime(mid_year, 12, 31)

        left_pmids = fetch_pmids_by_date_and_keywords(
            min_date=min_date,
            max_date=mid_dt.strftime("%Y/%m/%d"),
            antibiotic=antibiotic,
            pathogen=pathogen, 
            bacteria_synonym_df=bacteria_synonym_df,
            email=email,
            batch_size=batch_size,
            extra_clauses=extra_clauses
        )
        right_pmids = fetch_pmids_by_date_and_keywords(
            min_date=(mid_dt + timedelta(days=1)).strftime("%Y/%m/%d"),
            max_date=max_date,
            antibiotic=antibiotic,
            pathogen=pathogen, 
            bacteria_synonym_df=bacteria_synonym_df,
            email=email,
            batch_size=batch_size,
            extra_clauses=extra_clauses
        )
        return left_pmids + right_pmids
    return retrieve_pmids(query, webenv, query_key, count, batch_size=batch_size)


def update_metadata_cache(pmids, metadata_cache, email="your_email@example.com", batch_size=200):
    missing = [pmid for pmid in pmids if pmid not in metadata_cache]
    if not missing:
        return
    date_map, venue_map = fetch_pub_metadata(missing, email=email, batch_size=batch_size)
    for pmid in missing:
        metadata_cache[pmid] = {
            "date": date_map.get(pmid),
            "venue": venue_map.get(pmid, "Unknown")
        }


def get_metadata(pmid, metadata_cache):
    return metadata_cache.get(pmid, {"date": None, "venue": "Unknown"})


def add_venue_vectors(df, histogram_col="venue_histogram"):
    """Return a copy of df with a `venue_vector` column and the venue index map."""
    if histogram_col not in df.columns:
        raise KeyError(f"Column '{histogram_col}' not present in DataFrame")

    hist_series = df[histogram_col].fillna("{}")

    def _coerce_hist(value):
        if isinstance(value, dict):
            return value
        if isinstance(value, str):
            value = value.strip()
            if not value:
                return {}
            try:
                return json.loads(value)
            except json.JSONDecodeError:
                return {}
        return {}

    parsed = hist_series.apply(_coerce_hist)
    all_venues = sorted({venue for hist in parsed for venue in hist.keys()})
    venue_index = {venue: idx for idx, venue in enumerate(all_venues)}

    def _vectorize(hist):
        vector = [0] * len(all_venues)
        for venue, count in hist.items():
            idx = venue_index.get(venue)
            if idx is not None:
                vector[idx] = count
        return vector

    venue_vectors = parsed.apply(_vectorize)

    df_copy = df.copy()
    df_copy["venue_vector"] = venue_vectors
    return df_copy, venue_index


def plot_venue_vector(df_with_vecs, venue_index, row=None, ax=None, rotation=0, title=None, use_all_dates=False, pathogen=None, antibiotic=None, min_num=1):
    import matplotlib.pyplot as plt

    required_cols = {"venue_vector", "pathogen", "antibiotic"}
    missing = required_cols - set(df_with_vecs.columns)
    if missing:
        raise KeyError(f"DataFrame missing required columns: {missing}")

    if not isinstance(venue_index, dict):
        raise TypeError("venue_index must be a dictionary of venue->position.")

    if row is not None and (row < 0 or row >= len(df_with_vecs)):
        raise IndexError("Row index out of range for venue_vector plot.")

    if row is None:
        if pathogen is None or antibiotic is None:
            raise ValueError("Provide either `row` or both `pathogen` and `antibiotic`.")
        row_mask = (
            (df_with_vecs["pathogen"] == pathogen) &
            (df_with_vecs["antibiotic"] == antibiotic)
        )
        if not row_mask.any():
            raise ValueError("No rows match the specified pathogen/antibiotic combination.")
        current_row = df_with_vecs.loc[row_mask].iloc[0]
    else:
        current_row = df_with_vecs.iloc[row]
        if pathogen is None:
            pathogen = current_row["pathogen"]
        if antibiotic is None:
            antibiotic = current_row["antibiotic"]

    venues_sorted = [venue for venue, _ in sorted(venue_index.items(), key=lambda kv: kv[1])]

    def _coerce_vector(vector_like):
        if not isinstance(vector_like, (list, tuple)):
            raise TypeError("venue_vector entries must be list-like.")
        vec = list(vector_like)
        if len(vec) != len(venues_sorted):
            adjusted = [0] * len(venues_sorted)
            for venue, idx in venue_index.items():
                if idx < len(vec):
                    adjusted[idx] = vec[idx]
            vec = adjusted
        return vec

    if use_all_dates:
        mask = (
            (df_with_vecs["pathogen"] == pathogen) &
            (df_with_vecs["antibiotic"] == antibiotic)
        )
        grouped = df_with_vecs.loc[mask, "venue_vector"]
        aggregated = [0] * len(venues_sorted)
        for vector_like in grouped:
            try:
                vec = _coerce_vector(vector_like)
            except TypeError:
                continue
            for idx, value in enumerate(vec):
                aggregated[idx] += value
        vector = aggregated
    else:
        vector = _coerce_vector(current_row["venue_vector"])

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, max(4, len(venues_sorted) * 0.35)))
    else:
        fig = ax.figure

    positions = list(range(len(venues_sorted)))
    max_value = max(vector) if vector else 0
    offset = 0.02 * max_value if max_value else 0.1
    bars = ax.barh(positions, vector)
    for bar, value in zip(bars, vector):
        width = bar.get_width()
        y_pos = bar.get_y() + bar.get_height() / 2
        x_pos = width + offset if width else offset
        ax.text(x_pos, y_pos, f"{value}", va="center", ha="left", fontsize=8)
    ax.set_yticks(positions)
    ax.set_yticklabels(venues_sorted, rotation=rotation)
    ax.set_xlabel("Count")
    ax.set_ylabel("Venue")
    if title:
        ax.set_title(title)
    ax.margins(y=0.01)
    try:
        fig.tight_layout()
    except ValueError:
        pass
    return ax


def fetch_pmids_intervals(pathogens, antibiotics, bacteria_synonym_df, min_date, max_date, year_gap=5,
                          email="your_email@example.com", parquet_path="pmids_results.parquet",
                          save_every=50, extra_keywords=None, full_fetch_threshold=2000,
                          summary_batch_size=200):
    """
    For each (pathogen, antibiotic), query PubMed in year_gap intervals.
    If an interval exceeds PubMed's 9,999 cap, fetch_pmids_by_date_and_keywords
    automatically subdivides further.
    Skips queries already present in parquet_path (if it exists).
    Saves to parquet every `save_every` new tasks.
    Additional query clauses can be provided via `extra_keywords` (list of raw strings).
    Uses a single full-range fetch when total hits stay below `full_fetch_threshold`
    and partitions results locally via publication dates.
    """
    if os.path.exists(parquet_path):
        existing_df = pd.read_parquet(parquet_path)
        if "venue_histogram" not in existing_df.columns:
            existing_df["venue_histogram"] = None
    else:
        existing_df = pd.DataFrame(columns=["pathogen", "antibiotic", "start_date", "end_date", "pmids", "count", "venue_histogram"])

    overall_start_dt = datetime.strptime(min_date, "%Y/%m/%d")
    overall_end_dt = datetime.strptime(max_date, "%Y/%m/%d")

    total_tasks = len(pathogens) * len(antibiotics)
    task_counter = 0
    new_results = []
    metadata_cache = {}

    for ai, antibiotic in enumerate(antibiotics, start=1):
        for pi, pathogen in enumerate(pathogens, start=1):
            task_counter += 1
            task_progress = (task_counter / total_tasks) * 100
            
            synonyms = expand_synonyms(pathogen, bacteria_synonym_df)
            synonyms_list = list(synonyms) if synonyms is not None else []
            synonyms_str = ", ".join(map(str, synonyms_list)) if synonyms_list else "—"
            print(f"Processing antibiotic '{antibiotic}' | pathogen '{pathogen}' | synonyms: {synonyms_str}")


            interval_entries = []
            missing_entries = []

            for year in range(overall_start_dt.year, overall_end_dt.year + 1, year_gap):
                interval_start = f"{year}/01/01"
                interval_end_year = min(year + year_gap - 1, overall_end_dt.year)
                interval_end = f"{interval_end_year}/12/31"
                start_dt = datetime.strptime(interval_start, "%Y/%m/%d")
                end_dt = datetime.strptime(interval_end, "%Y/%m/%d")

                mask = (
                    (existing_df["pathogen"] == pathogen) &
                    (existing_df["antibiotic"] == antibiotic) &
                    (existing_df["start_date"] == interval_start) &
                    (existing_df["end_date"] == interval_end)
                )

                entry = {
                    "start": interval_start,
                    "end": interval_end,
                    "start_dt": start_dt,
                    "end_dt": end_dt,
                    "cached": False,
                    "count": 0
                }

                if mask.any():
                    entry["cached"] = True
                    entry["count"] = int(existing_df.loc[mask, "count"].iloc[0])
                    print(
                        f"[{task_counter:>3}/{total_tasks}] "
                        f"({task_progress:5.1f}%) | "
                        f"{pathogen:<40} | {antibiotic:<20} | "
                        f"{interval_start} – {interval_end} | "
                        f"{entry['count']:5} PMIDs (cached)"
                    )
                else:
                    missing_entries.append(entry)
                interval_entries.append(entry)

            if not missing_entries:
                continue

            query, total_count, webenv, query_key = search_pmids(
                antibiotic,
                pathogen,
                bacteria_synonym_df,
                extra_clauses=extra_keywords,
                min_date=min_date,
                max_date=max_date,
                email=email,
                
            )

            if total_count == 0:
                for entry in missing_entries:
                    new_results.append({
                        "pathogen": pathogen,
                        "antibiotic": antibiotic,
                        "start_date": entry["start"],
                        "end_date": entry["end"],
                        "pmids": [],
                        "count": 0,
                        "venue_histogram": json.dumps({})
                    })
                    print(
                        f"[{task_counter:>3}/{total_tasks}] "
                        f"({task_progress:5.1f}%) | "
                        f"{pathogen:<40} | {antibiotic:<20} | "
                        f"{entry['start']} – {entry['end']} | "
                        f"{0:5} PMIDs (new)"
                    )
                if len(new_results) >= save_every:
                    df_new = pd.DataFrame(new_results)
                    existing_df = merge_with_existing(df_new, parquet_path=parquet_path)
                    new_results = []
                continue

            if total_count <= full_fetch_threshold:
                pmids_all = retrieve_pmids(query, webenv, query_key, total_count)
                update_metadata_cache(pmids_all, metadata_cache, email=email, batch_size=summary_batch_size)

                for entry in missing_entries:
                    pmids_interval = []
                    venue_counter = Counter()
                    for pmid in pmids_all:
                        meta = get_metadata(pmid, metadata_cache)
                        pub_date = meta["date"]
                        if not pub_date:
                            continue
                        if not (overall_start_dt <= pub_date <= overall_end_dt):
                            continue
                        if entry["start_dt"] <= pub_date <= entry["end_dt"]:
                            pmids_interval.append(pmid)
                            venue_value = meta["venue"] or "Unknown"
                            venue_counter[venue_value] += 1
                    histogram_json = json.dumps(dict(venue_counter))
                    new_results.append({
                        "pathogen": pathogen,
                        "antibiotic": antibiotic,
                        "start_date": entry["start"],
                        "end_date": entry["end"],
                        "pmids": pmids_interval,
                        "count": len(pmids_interval),
                        "venue_histogram": histogram_json
                    })
                    print(
                        f"[{task_counter:>3}/{total_tasks}] "
                        f"({task_progress:5.1f}%) | "
                        f"{pathogen:<40} | {antibiotic:<20} | "
                        f"{entry['start']} – {entry['end']} | "
                        f"{len(pmids_interval):5} PMIDs (full fetch partitioned)"
                    )
                    if len(new_results) >= save_every:
                        df_new = pd.DataFrame(new_results)
                        existing_df = merge_with_existing(df_new, parquet_path=parquet_path)
                        new_results = []
                continue

            for entry in missing_entries:
                pmids_interval = fetch_pmids_by_date_and_keywords(
                    min_date=entry["start"],
                    max_date=entry["end"],
                    antibiotic=antibiotic,
                    pathogen=pathogen, 
                    bacteria_synonym_df=bacteria_synonym_df,
                    email=email,
                    extra_clauses=extra_keywords
                )
                update_metadata_cache(pmids_interval, metadata_cache, email=email, batch_size=summary_batch_size)
                venue_counter = Counter()
                filtered_pmids = []
                for pmid in pmids_interval:
                    meta = get_metadata(pmid, metadata_cache)
                    pub_date = meta["date"]
                    if pub_date and not (entry["start_dt"] <= pub_date <= entry["end_dt"]):
                        continue
                    filtered_pmids.append(pmid)
                    venue_value = meta["venue"] or "Unknown"
                    venue_counter[venue_value] += 1
                histogram_json = json.dumps(dict(venue_counter))
                new_results.append({
                    "pathogen": pathogen,
                    "antibiotic": antibiotic,
                    "start_date": entry["start"],
                    "end_date": entry["end"],
                    "pmids": filtered_pmids,
                    "count": len(filtered_pmids),
                    "venue_histogram": histogram_json
                })
                print(
                    f"[{task_counter:>3}/{total_tasks}] "
                    f"({task_progress:5.1f}%) | "
                    f"{pathogen:<40} | {antibiotic:<20} | "
                    f"{entry['start']} – {entry['end']} | "
                    f"{len(filtered_pmids):5} PMIDs (new)"
                )
                if len(new_results) >= save_every:
                    df_new = pd.DataFrame(new_results)
                    existing_df = merge_with_existing(df_new, parquet_path=parquet_path)
                    new_results = []

    if new_results:
        df_new = pd.DataFrame(new_results)
        existing_df = merge_with_existing(df_new, parquet_path=parquet_path)

    return existing_df



---

In [4]:
antibiotics_df = pd.read_excel("antibiotic_list.xlsx")
antibiotics_list = antibiotics_df['Antibiotic'].values
print(f'#antibiotics: {len(antibiotics_list)}')

data_df = pd.read_parquet("data.parquet")
pathogens_list = data_df[data_df['n_causation']>=3]['CanonicalSpecies'].values
print(f'#pathogens: {len(pathogens_list)}')

bacteria_synonym_df = pd.read_csv("bacteria_synonym_lookup.csv")
print(f'#synonyms: {len(bacteria_synonym_df)}')

#antibiotics: 245
#pathogens: 773
#synonyms: 24932


In [5]:
if False:
    pathogens_list = ['Staphylococcus argenteus',
    'Staphylococcus aureus', 'Staphylococcus auricularis',
    'Staphylococcus capitis', 'Staphylococcus caprae',
    'Staphylococcus cohnii', 'Staphylococcus condimenti',
    'Staphylococcus epidermidis', 'Staphylococcus gallinarum',
    'Staphylococcus haemolyticus', 'Staphylococcus hominis',
    'Staphylococcus hyicus', 'Staphylococcus intermedius',
    'Staphylococcus lugdunensis', 'Staphylococcus pasteuri',
    'Staphylococcus pettenkoferi', 'Staphylococcus pseudintermedius',
    'Staphylococcus saccharolyticus', 'Staphylococcus saprophyticus',
    'Staphylococcus schleiferi', 'Staphylococcus simulans',
    'Staphylococcus warneri', 'Staphylococcus xylosus',
    'Neisseria animaloris', 'Neisseria bacilliformis',
    'Neisseria canis', 'Neisseria cinerea', 'Neisseria elongata',
    'Neisseria flava', 'Neisseria flavescens', 'Neisseria gonorrhoeae',
    'Neisseria lactamica', 'Neisseria macacae',
    'Neisseria meningitidis', 'Neisseria mucosa', 'Neisseria perflava',
    'Neisseria sicca', 'Neisseria subflava', 'Neisseria weaveri']

In [6]:
if False:
    pathogens_list = ["Escherichia coli", "Staphylococcus aureus", "Neisseria canis"]
    antibiotics_list = ["Piperacillin/tazobactam", "Cefteram-pivoxil"]

In [ ]:
parquet_path = "pmids_results.parquet"

df = fetch_pmids_intervals(
    pathogens=pathogens_list,
    antibiotics=antibiotics_list,
    bacteria_synonym_df=bacteria_synonym_df,
    min_date="1975/01/01",
    max_date="2024/12/31",
    year_gap=5,
    extra_keywords=[
        "(resistant OR resistance OR nonsusceptible OR non-susceptible)"
    ],
    parquet_path=parquet_path
)
#"NOT (susceptible OR susceptibility OR sensitive OR sensitivity)"
merge_with_existing(df, parquet_path=parquet_path)

In [ ]:
df = pd.read_parquet("pmids_results.parquet")
df

In [5]:
df_with_vecs, venue_index = add_venue_vectors(df)
venue_index_df = pd.Series({pos: venue for venue, pos in venue_index.items()}, name="venue").to_frame()
venue_index_df.to_parquet('venue_index.parquet', index=False)
df_with_vecs.to_parquet('pmids_results_venues.parquet', index=False)
df_with_vecs

,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram,venue_vector
0,Abiotrophia defectiva,Spectinomycin,1975/01/01,1979/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Abiotrophia defectiva,Spectinomycin,1980/01/01,1984/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Abiotrophia defectiva,Spectinomycin,1985/01/01,1989/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Abiotrophia defectiva,Spectinomycin,1990/01/01,1994/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Abiotrophia defectiva,Spectinomycin,1995/01/01,1999/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...
1893845,Yokenella regensburgei,Trimethoprim,2000/01/01,2004/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1893846,Yokenella regensburgei,Trimethoprim,2005/01/01,2009/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1893847,Yokenella regensburgei,Trimethoprim,2010/01/01,2014/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1893848,Yokenella regensburgei,Trimethoprim,2015/01/01,2019/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


---